# cjm-transcript-correction-core

> A frontend-agnostic core for the transcript correction workflow — the first downstream graph-extension core; composes the graph-storage capability worker into a headless pipeline that applies unified, non-destructive corrections (text, punctuation, segmentation) as a supersede-able overlay on a committed decomposition spine, recomputes the review worklist from deterministic signals, and exposes a CLI as its first driver.

## Install

```bash
pip install cjm_transcript_correction_core
```

## Project Structure

```
nbs/
├── cli.ipynb      # The CLI driver -- the correction core's first (and currently only) frontend.
├── graph.ipynb    # The correction overlay's graph I/O: targeted (scale-shaped) reads of a committed
├── models.ipynb   # Overlay data shapes for the transcript-correction workflow: the Correction /
├── pipeline.ipynb # The headless correction workflow: load a decomp run manifest, resolve the shared
└── signals.ipynb  # Pure deterministic Tier-1 signal functions (no capability calls): empty-segment
```

Total: 5 notebooks

## Module Dependencies

```mermaid
graph LR
    cli["cli<br/>cli"]
    graph_mod["graph<br/>graph"]
    models["models<br/>models"]
    pipeline["pipeline<br/>pipeline"]
    signals["signals<br/>signals"]

    cli --> pipeline
    cli --> models
    graph_mod --> models
    pipeline --> models
    pipeline --> graph_mod
    pipeline --> signals
    signals --> models
```

*7 cross-module dependencies detected*

## CLI Reference

### `cjm-transcript-correction-core` Command

```
usage: cjm-transcript-correction-core [-h] {run,review} ...

Headless transcript correction: non-destructive overlay on a committed source
spine.

positional arguments:
  {run,review}
    run         Prune empty segments + surface the worklist (deterministic)
    review      Interactive text-correction review of the flagged worklist

options:
  -h, --help    show this help message and exit

```

For detailed help on any command, use `cjm-transcript-correction-core <command> --help`.

## Module Overview

Detailed documentation for each module in the project:

### cli (`cli.ipynb`)
> The CLI driver -- the correction core's first (and currently only) frontend.

#### Import

```python
from cjm_transcript_correction_core.cli import (
    logger,
    build_parser,
    load_capabilities,
    run_command,
    review_command,
    main
)
```

#### Functions

```python
def _add_common_run_args(p: argparse.ArgumentParser) -> None:  # Shared run/review arguments
    "Attach the capability / session / output arguments shared by `run` and `review`."
```

```python
def build_parser() -> argparse.ArgumentParser:  # Configured CLI parser
    """
    Build the CLI parser (subcommands: run, review).
    
    Stage 5: --secondary-manifest is RETIRED — the cross-transcriber diff is
    intra-graph now (variant slices on the shared-skeleton segments).
    """
```

```python
def load_capabilities(
    manager: CapabilityManager,                      # Freshly constructed manager
    instance_ids: List[str],                     # Capability names to load, in order
    configs: Optional[Dict[str, Dict]] = None,   # Per-capability load-time config (e.g. graph db_path)
) -> None
    "Discover manifests + load each capability, passing per-capability config (CR-2 caller-wins)."
```

```python
async def run_command(
    args: argparse.Namespace,  # Parsed args for the `run` subcommand
) -> int:  # Process exit code
    "Execute the `run` subcommand: correct a decomp manifest's committed spine."
```

```python
async def review_command(
    args: argparse.Namespace,  # Parsed args for the `review` subcommand
) -> int:  # Process exit code
    "Execute the `review` subcommand: interactive text corrections over the flagged worklist."
```

```python
def main(
    argv: Optional[List[str]] = None,  # Argument list override (None = sys.argv)
) -> int:  # Process exit code
    "CLI entry point (console script: `cjm-transcript-correction-core`)."
```


### graph (`graph.ipynb`)
> The correction overlay's graph I/O: targeted (scale-shaped) reads of a committed

#### Import

```python
from cjm_transcript_correction_core.graph import (
    submit_and_wait,
    source_audio_segment_ids,
    resolve_source_renditions,
    load_source_segments,
    load_empty_segments,
    count_source_segments,
    build_correction_node,
    build_prune_correction,
    commit_nodes_edges,
    start_session,
    get_session,
    set_session_status,
    record_review_markers,
    load_review_markers,
    find_corrections_for_session,
    find_prior_corrections_by_hash,
    corrections_to_edits,
    project_effective_spine,
    build_text_correction,
    commit_text_correction,
    active_corrections,
    load_source_corrections,
    find_active_text_corrections_batch,
    find_active_text_correction,
    load_variant_texts
)
```

#### Functions

```python
async def submit_and_wait(
    queue: JobQueue,                  # Started job queue
    instance_id: str,                 # Capability instance to invoke
    *,
    timeout: Optional[float] = None,  # Seconds to wait; None = no limit
    **kwargs,                         # Forwarded to the capability action
) -> Any:  # Completed job result payload
    """
    Submit one capability job, wait for it, return its result (raise on failure).
    
    (Restored as its own cell after the stage-2 field_of retirement removed
    the shared cell that bundled both functions — the loop-back harness
    caught the casualty; one-fn-per-cell prevents the recurrence.)
    """
```

```python
def _row_to_spine_segment(row: Dict[str, Any]) -> SpineSegment:  # One projected row -> SpineSegment
    """Build a SpineSegment from a projected row (audio anchor + per-transcriber slices)."""
    sources = row.get("sources") or []
    text_from = row.get("text_from")
    audio = next((s for s in sources if (s.get("slice") or {}).get("kind") == "time"), None)
    slices: List[Dict[str, Any]] = []
    "Build a SpineSegment from a projected row (audio anchor + per-transcriber slices)."
```

```python
async def source_audio_segment_ids(
    queue: JobQueue,  # Started job queue
    graph_id: str,    # Graph-storage capability id
    source_id: str,   # Source node id
) -> List[str]:  # Ordered AudioSegment node ids under the Source
    "The Source's coarse spine (one small typed read; ordered by index)."
```

```python
def _rendition_label(r: Dict[str, Any]) -> str:  # Human-readable rendition label
    """A rendition's selector/display label ("raw" or its preprocessing chain)."""
    if r.get("is_raw")
    "A rendition's selector/display label ("raw" or its preprocessing chain)."
```

```python
async def resolve_source_renditions(
    queue: JobQueue,                   # Started job queue
    graph_id: str,                     # Graph-storage capability id
    source_id: str,                    # Source node id
    selector: Optional[str] = None,    # Which rendition: "raw" | a preprocessing-descriptor substring | None = auto
) -> List[str]:  # The AudioRendition ids whose fine spine to operate on (one chain group)
    """
    Pick the AudioRendition set whose fine Segment spine correction operates on.
    
    The fine spine hangs under renditions (raw | vocals | ...) that COEXIST under
    one AudioSegment. With ONE decomposed rendition (the common case) it is
    selected automatically; multiple decomposed renditions REQUIRE an explicit
    `selector` ("raw", or a substring of the preprocessing descriptor) — the
    spines are never silently mixed. Returns the rendition ids of the chosen
    chain group (one per AudioSegment), or [] when nothing is decomposed yet.
    """
```

```python
async def _populated_rendition_ids(
    queue: JobQueue,          # Started job queue
    graph_id: str,            # Graph-storage capability id
    rendition_ids: List[str], # Candidate rendition ids
) -> set:  # Subset that own >=1 fine Segment
    "Which renditions actually carry a fine Segment spine (one batched read)."
```

```python
def _spine_query(
    rendition_ids: List[str],  # The AudioRendition ids the spine hangs under
    **overrides,               # NodeQuery field overrides (where / count / limit / ...)
) -> NodeQuery:  # The source-spine read
    "Segments PART_OF the chosen AudioRenditions (batched far-end), ordered by index."
```

```python
async def load_source_segments(
    queue: JobQueue,                        # Started job queue
    graph_id: str,                          # Graph-storage capability id
    source_id: str,                         # Source node id
    limit: Optional[int] = None,            # Optional page size
    offset: Optional[int] = None,           # Optional page offset
    rendition_selector: Optional[str] = None,  # Which rendition spine (None = auto-select)
) -> List[SpineSegment]:  # Ordered spine segments (by index)
    "Load a Source's fine Segment spine under its chosen rendition (typed query surface)."
```

```python
async def load_empty_segments(
    queue: JobQueue,  # Started job queue
    graph_id: str,    # Graph-storage capability id
    source_id: str,   # Source node id
    rendition_selector: Optional[str] = None,  # Which rendition spine (None = auto-select)
) -> List[SpineSegment]:  # Only the empty-text segments (server-side filtered)
    """
    Load ONLY a Source's empty-text segments under its chosen rendition (D14 prune).
    
    The evidenced OR case (text = '' OR text IS NULL) = TWO server-side-filtered
    queries unioned client-side — compound boolean predicates stay deferred (P8);
    both halves materialize ~10% of the spine, never the whole source.
    """
```

```python
async def count_source_segments(
    queue: JobQueue,  # Started job queue
    graph_id: str,    # Graph-storage capability id
    source_id: str,   # Source node id
    rendition_selector: Optional[str] = None,  # Which rendition spine (None = auto-select)
) -> int:  # Number of fine Segment nodes under the Source's chosen rendition
    "Count a Source's segments server-side under its chosen rendition (typed count mode)."
```

```python
def _edge(
    source_id: str,                            # Origin node id
    target_id: str,                            # Destination node id
    relation_type: str,                        # Edge relation type
    properties: Optional[Dict[str, Any]] = None,  # Edge properties
) -> Dict[str, Any]:  # Edge wire dict
    """
    Build an edge wire dict with a GENERATED id.
    
    Used for REVIEWED markers only: review decisions are EVENTS (a re-decision
    appends), so their edges keep generated ids. Structural overlay edges
    (CORRECTS / SUPERSEDES / DERIVED_FROM) use the layer's deterministic
    `make_edge` — unique per (source, target, relation) by construction.
    """
```

```python
def build_correction_node(
    correction_type: str,                  # "text_content" | "punctuation" | "grouping"
    session_id: str,                       # Owning session id
    payload: Dict[str, Any],               # Type-specific payload
    actor: str = "human",                  # Actor
    status: str = "applied",               # Lifecycle status
    canonical_form: Optional[str] = None,  # Optional entity key
    rationale: Optional[str] = None,       # Optional note
) -> Correction:  # The Correction overlay node (not yet committed)
    "Construct a Correction overlay node (pure; commit happens separately)."
```

```python
def build_prune_correction(
    source_id: str,              # Source being corrected
    pruned: List[SpineSegment],  # Empty layer-0 segments to prune
    session_id: str,             # Owning session id
    actor: str = "human",        # Actor
) -> Tuple[Dict[str, Any], List[Dict[str, Any]]]:  # (correction node dict, edge dicts)
    """
    Build one batch grouping Correction that prunes empty segments (D14).
    
    Non-destructive: layer-0 nodes are NOT deleted; the Correction records the
    pruned ids and DERIVED_FROM edges point at each pruned Segment. The effective
    spine drops them at projection time (reversible by superseding this node).
    The payload carries the layer's `prune` spine-edit op vocabulary.
    """
```

```python
async def commit_nodes_edges(
    queue: JobQueue,              # Started job queue
    graph_id: str,                # Graph-storage capability id
    nodes: List[Dict[str, Any]],  # Node wire dicts
    edges: List[Dict[str, Any]],  # Edge wire dicts
) -> Dict[str, int]:  # {"nodes": n, "edges": m} created counts
    """
    Commit overlay nodes/edges through the layer's idempotent extend_graph.
    
    Stage 5 (C5 plumbing migrated): the layer owns emit-if-absent +
    verify-if-present; overlay nodes have generated ids so they always add,
    but a replayed commit collides into a verified no-op instead of
    duplicating structural edges.
    """
```

```python
async def start_session(
    queue: JobQueue,   # Started job queue
    graph_id: str,     # Graph-storage capability id
    scope: List[str],  # Source node ids in scope
) -> CorrectionSession:  # The committed CorrectionSession
    "Create + commit a new CorrectionSession node."
```

```python
async def get_session(
    queue: JobQueue,  # Started job queue
    graph_id: str,    # Graph-storage capability id
    session_id: str,  # CorrectionSession node id
) -> Optional[Dict[str, Any]]:  # The session node dict, or None
    "Fetch a CorrectionSession node by id (resume/reopen) — typed get, dict shape preserved."
```

```python
async def set_session_status(
    queue: JobQueue,  # Started job queue
    graph_id: str,    # Graph-storage capability id
    session_id: str,  # CorrectionSession node id
    status: str,      # New status ("in_progress" | "completed" | "reopened")
) -> None
    """
    Update a session's status + updated_at.
    
    The ONLY update_node use in this core: a CorrectionSession is OVERLAY metadata
    whose lifecycle is mutable. Layer-0 Segments + Corrections stay append-only.
    """
```

```python
async def record_review_markers(
    queue: JobQueue,                   # Started job queue
    graph_id: str,                     # Graph-storage capability id
    session_id: str,                   # Owning session id
    decisions: List[Tuple[str, str]],  # (segment_id, decision) pairs
) -> int:  # Number of REVIEWED edges committed
    "Persist per-(session, segment) review markers as REVIEWED edges."
```

```python
async def load_review_markers(
    queue: JobQueue,  # Started job queue
    graph_id: str,    # Graph-storage capability id
    session_id: str,  # Owning session id
) -> Dict[str, str]:  # segment_id -> decision for the session
    "Load a session's review markers (typed edge projection over REVIEWED edges)."
```

```python
def _node_to_correction_dict(
    node: Any,  # GraphNode (typed task result) or its wire dict
) -> Dict[str, Any]:  # Correction properties + "id" (the pre-stage-4 row shape)
    "Flatten a Correction node to its properties dict + id."
```

```python
async def find_corrections_for_session(
    queue: JobQueue,  # Started job queue
    graph_id: str,    # Graph-storage capability id
    session_id: str,  # Owning session id
) -> List[Dict[str, Any]]:  # Correction property dicts for the session
    "List corrections recorded in a session (typed property filter)."
```

```python
async def find_prior_corrections_by_hash(
    queue: JobQueue,    # Started job queue
    graph_id: str,      # Graph-storage capability id
    content_hash: str,  # SourceRef content hash to look up
) -> List[Dict[str, Any]]:  # Corrections whose CORRECTS target carried this hash
    """
    Cross-transcript correction-cache lookup (targeted; the graph IS the lexicon).
    
    THE stage-4 promotion landing: the raw two-table JOIN this function carried
    (C-ledger site 6) became ONE typed far-end provenance constraint
    (`RelationPredicate.node_source`, content-hash-primary per CR-19) — the
    hottest review-path read is portable now.
    """
```

```python
def corrections_to_edits(
    corrections: List[Dict[str, Any]],  # ACTIVE correction property dicts
) -> List[SpineEdit]:  # The layer's neutral spine-edit operations
    """
    Map this core's Correction payloads onto the layer's spine-edit vocabulary.
    
    The DOMAIN interpretation (correction_type + payload shapes) stays here;
    the projection MECHANICS (ordering, latest-wins, prune/replace/boundary
    semantics) are the layer's (CR-18: C11 migrated).
    """
```

```python
def project_effective_spine(
    segments: List[SpineSegment],       # Ordered layer-0 spine
    corrections: List[Dict[str, Any]],  # Applied correction property dicts
) -> List[SpineSegment]:  # The effective spine after applying corrections
    """
    Project the effective spine = layer-0 + applied corrections.
    
    Stage 5: the projection MECHANICS live in `cjm-context-graph-layer`
    (`project_effective_spine` over SpineUnits — prune, replace_text, and the
    reserved boundary_shift, with created_at ordering + latest-wins); this
    wrapper converts SpineSegments <-> SpineUnits and re-attaches the
    segment metadata by id.
    """
```

```python
def build_text_correction(
    source_id: str,                        # Source the segment belongs to
    segment_id: str,                       # Layer-0 Segment being corrected
    new_text: str,                         # Corrected text
    session_id: str,                       # Owning session id
    old_text: Optional[str] = None,        # Prior effective text (for the record)
    supersedes_id: Optional[str] = None,   # Prior Correction this one replaces (re-edit)
    actor: str = "human",                  # Actor
    canonical_form: Optional[str] = None,  # Optional entity key (cross-transcript matching)
    rationale: Optional[str] = None,       # Optional note
) -> Tuple[Dict[str, Any], List[Dict[str, Any]]]:  # (correction node dict, edge dicts)
    """
    Build a text_content Correction + its CORRECTS (+ optional SUPERSEDES) edges.
    
    Non-destructive: the layer-0 Segment is unchanged; the correction carries the
    new text in its payload + a CORRECTS edge to the segment. A re-edit adds a
    SUPERSEDES edge (new -> prior) so supersession is graph-native + append-only
    (the prior Correction is never mutated; it is excluded from the effective view
    because it is a SUPERSEDES target — the C16 semantics, layer-resolved).
    """
```

```python
async def commit_text_correction(
    queue: JobQueue,                       # Started job queue
    graph_id: str,                         # Graph-storage capability id
    source_id: str,                        # Source the segment belongs to
    segment_id: str,                       # Layer-0 Segment being corrected
    new_text: str,                         # Corrected text
    session_id: str,                       # Owning session id
    old_text: Optional[str] = None,        # Prior effective text
    supersedes_id: Optional[str] = None,   # Prior Correction to supersede (re-edit)
    actor: str = "human",                  # Actor
    canonical_form: Optional[str] = None,  # Optional entity key
) -> str:  # The new Correction node id
    "Commit a text_content correction (node + CORRECTS [+ SUPERSEDES]) + a REVIEWED marker."
```

```python
def active_corrections(
    corrections: List[Dict[str, Any]],  # Corrections (e.g. from load_source_corrections)
    superseded_ids: set,                # Ids that are SUPERSEDES targets
) -> List[Dict[str, Any]]:  # Only the effective (non-superseded) corrections
    "Filter to the effective correction set (the layer's resolve_active over a read superseded set)."
```

```python
async def _superseded_ids(
    queue: JobQueue,            # Started job queue
    graph_id: str,              # Graph-storage capability id
    correction_ids: List[str],  # Candidate correction ids
) -> set:  # Subset that are SUPERSEDES targets
    "Which of the given corrections are SUPERSEDES targets (typed id-list batch)."
```

```python
async def load_source_corrections(
    queue: JobQueue,  # Started job queue
    graph_id: str,    # Graph-storage capability id
    source_id: str,   # Source node id
) -> Tuple[List[Dict[str, Any]], set]:  # (all corrections for the source, superseded id set)
    """
    Load every Correction targeting a Source (across sessions) + the superseded-id set.
    
    Source-scoped (corrections carry payload.source_id — a dotted-path typed
    predicate) so the effective view is a property of the SOURCE, not one
    session — the persistence/resume/reopen requirement. Append-only:
    supersession is read from SUPERSEDES edges, never a status mutation.
    """
```

```python
async def find_active_text_corrections_batch(
    queue: JobQueue,         # Started job queue
    graph_id: str,           # Graph-storage capability id
    segment_ids: List[str],  # Segments to look up
) -> Dict[str, Dict[str, Any]]:  # segment_id -> active text correction
    """
    Active text corrections for MANY segments in TWO round-trips (C17).
    
    One far-end batch read (Corrections with CORRECTS edges into the id set;
    `RelationPredicate.node_ids`) + one superseded-set read — replacing the
    per-item lookup the review loop paid (a 1,275-item review would have been
    1,275 queries; now 2). Latest non-superseded correction per segment wins.
    """
```

```python
async def find_active_text_correction(
    queue: JobQueue,  # Started job queue
    graph_id: str,    # Graph-storage capability id
    segment_id: str,  # Segment to look up
) -> Optional[Dict[str, Any]]:  # The current non-superseded text correction, or None
    "Single-segment convenience over the batch read (cross-session; latest wins)."
```

```python
async def load_variant_texts(
    queue: JobQueue,               # Started job queue
    graph_id: str,                 # Graph-storage capability id
    segments: List[SpineSegment],  # Spine segments (with text_slices)
) -> Dict[str, Dict[str, str]]:  # segment_id -> {transcriber: chunk text}
    """
    Resolve per-transcriber chunk texts from the segments' CharSlice refs.
    
    Stage 5: the cross-transcriber diff is INTRA-GRAPH — text is stored once
    per transcriber at the coarse Transcript nodes; this fetches ALL referenced
    Transcript nodes in ONE batched typed read and slices their text
    client-side by each segment's char ranges. Replaces the retired
    second-decomp-manifest positional join (C4/C14's shared-skeleton model).
    """
```

#### Variables

```python
_SPINE_PROJECTION = [6 items]  # + structural "id"
```

### models (`models.ipynb`)
> Overlay data shapes for the transcript-correction workflow: the Correction /

#### Import

```python
from cjm_transcript_correction_core.models import (
    CorrectionRelations,
    Correction,
    CorrectionSession,
    SpineSegment,
    WorklistItem,
    CorrectionConfig,
    CorrectionManifest,
    new_run_id
)
```

#### Functions

```python
def new_run_id() -> str:  # e.g. "correct_20260608_153000_1a2b3c4d"
    "Generate a unique, sortable correction run id."
```

#### Classes

```python
class CorrectionRelations:
    "Registry of edge types the correction overlay adds to the spine graph."
    
    def all(cls) -> list:  # All relation type strings
        "Return all defined relation types."
```

```python
@dataclass
class Correction:
    """
    A single non-destructive correction over the committed spine (overlay node).
    
    Layer-0 spine nodes are immutable; every correction is a supersede-able
    overlay. Defined IN-CORE (the C6 pattern, kept at stage 5 after
    cjm-graph-domains dissolved): a plain dataclass mapping itself onto the
    generic GraphNode. Corrections are DECISIONS (asserted events) — they keep
    GENERATED ids, the FLIP-TRIGGER-protected class.
    """
    
    correction_type: str  # "text_content" | "punctuation" | "grouping"
    status: str = 'applied'  # "proposed" | "applied" | "superseded"
    session_id: str = ''  # Owning CorrectionSession id
    payload: Dict[str, Any] = field(...)  # Type-specific data (new text, prune set, ...)
    actor: str = 'human'  # "human" | "agent:<id>" | "capability:<name>"
    canonical_form: Optional[str]  # Optional entity key (cross-transcript matching)
    rationale: Optional[str]  # Optional human/agent note
    created_at: float = field(...)  # Unix timestamp
    id: str = field(...)  # Generated node id (decision = event)
    
    def to_graph_node(self) -> GraphNode:  # Generic graph node (label = class name)
            """Map onto a generic GraphNode (None-valued fields excluded from properties)."""
            props = {k: v for k, v in asdict(self).items() if k != "id" and v is not None}
        "Map onto a generic GraphNode (None-valued fields excluded from properties)."
```

```python
@dataclass
class CorrectionSession:
    "A resumable, reopen-able correction review over one or more sources."
    
    status: str = 'in_progress'  # "in_progress" | "completed" | "reopened"
    scope: List[str] = field(...)  # Source node ids in scope
    started_at: float = field(...)  # Unix timestamp at session start
    updated_at: float = field(...)  # Unix timestamp of last activity
    id: str = field(...)  # Generated node id (session = event)
    
    def to_graph_node(self) -> GraphNode:  # Generic graph node
            """Map onto a generic GraphNode (None-valued fields excluded from properties)."""
            props = {k: v for k, v in asdict(self).items() if k != "id" and v is not None}
        "Map onto a generic GraphNode (None-valued fields excluded from properties)."
```

```python
@dataclass
class SpineSegment:
    """
    A committed layer-0 Segment loaded from the graph (read view).
    
    Stage 5 (Source-rooted schema): segments carry an audio `TimeSlice` ref
    (the stable anchor) + per-transcriber `CharSlice` refs into Transcript
    nodes; `content_hash` is the AUTHORITATIVE text's hash (the `text_from`
    slice) — the cross-transcript cache key.
    """
    
    id: str  # Graph Segment node id
    index: int  # 0-based position in the source spine
    text: str  # Layer-0 text (may be empty for silence VAD chunks)
    start_time: Optional[float]  # Source-coordinate start (seconds)
    end_time: Optional[float]  # Source-coordinate end (seconds)
    source_locator: Optional[str]  # Audio SourceRef locator URI (the stable provenance anchor)
    content_hash: Optional[str]  # Authoritative text slice's content_hash (None when empty)
    text_from: Optional[str]  # Authoritative Transcript node id (provenance designation)
    text_slices: List[Dict[str, Any]] = field(...)  # [{transcript, start, end, content_hash}]
    
    def is_empty(self) -> bool:  # True when the segment has no non-whitespace text
        "Empty-text segment (silence VAD chunk with no aligned words; decomp D14)."
```

```python
@dataclass
class WorklistItem:
    "One spine segment surfaced for review, with its deterministic Tier-1 flags."
    
    segment: SpineSegment  # The segment under review
    flags: List[str] = field(...)  # Tier-1 signal flags (empty, boundary, divergence, ...)
    
    def index(self) -> int:  # Segment spine index
        "Spine index of the underlying segment."
```

```python
@dataclass
class CorrectionConfig:
    "Configuration for one correction run."
    
    graph_capability: str = 'cjm-capability-graph-sqlite'  # Graph-storage capability id
    graph_db_path: Optional[str]  # Graph DB the spine lives in (from the decomp manifest)
    actor: str = 'human'  # Actor recorded on corrections + review markers
    assume_yes: bool = False  # Auto-accept HITL seams (headless mode)
    prune_empty: bool = True  # Run the D14 empty-segment prune as the first operation
    rendition_selector: Optional[str]  # Which AudioRendition spine to correct ("raw" | preprocessing substring); None = auto-select the populated one (error if ambiguous)
    
    def to_dict(self) -> Dict[str, Any]:  # Plain-dict snapshot for the manifest
        "Serialize to a plain dict."
```

```python
@dataclass
class CorrectionManifest:
    """
    Durable record of one correction run (proto-bundle; chainable, CR-20).
    
    Schema 0.2.0 (stage 5): `documents` became `sources` (Document dissolved
    into Source); the cross-transcriber diff is intra-graph now, so the
    secondary-manifest pointer is gone.
    """
    
    run_id: str  # Unique run identifier
    created_at: float  # Unix timestamp at run start
    config: Dict[str, Any]  # CorrectionConfig snapshot
    decomp_manifest: str  # Path to the consumed decomp run manifest
    graph_db_path: str  # The shared graph DB the spine + overlay live in
    session_id: str  # CorrectionSession node id this run used
    source_format: str = ''  # Upstream manifest format tag (interchange contract)
    source_version: str = ''  # Upstream manifest schema version
    signals_used: List[str] = field(...)  # Deterministic signals active this run
    sources: List[Dict[str, Any]] = field(...)  # Per-source outcome records
    FORMAT: str = field(...)  # Format tag
    VERSION: str = field(...)  # Schema version
    
    def to_dict(self) -> Dict[str, Any]:  # Plain-dict form for JSON serialization
            """Serialize to a plain dict."""
            return {
                "format": self.FORMAT,
        "Serialize to a plain dict."
    
    def save(
            self,
            path: Union[str, Path],  # Destination JSON file (parent dirs created)
        ) -> Path:  # The written path
        "Write the manifest as pretty-printed JSON."
```


### pipeline (`pipeline.ipynb`)
> The headless correction workflow: load a decomp run manifest, resolve the shared

#### Import

```python
from cjm_transcript_correction_core.pipeline import (
    logger,
    load_decomp_manifest,
    resolve_graph_db_path,
    compute_worklist,
    confirm_seam,
    prune_empty_segments,
    collect_capability_info,
    run_correction,
    review_worklist,
    run_review
)
```

#### Functions

```python
def load_decomp_manifest(
    path: str,  # Path to a decomp-core run manifest JSON
) -> Dict[str, Any]:  # Parsed manifest dict
    "Load + lightly validate a decomp-core run manifest (untyped JSON; CR-20 interchange)."
```

```python
def resolve_graph_db_path(
    manifest: Dict[str, Any],        # Decomp manifest dict
    graph_capability: str,               # Graph-storage capability id
    override: Optional[str] = None,  # Explicit override (wins)
) -> Optional[str]:  # Absolute DB path the spine lives in
    """
    Resolve the graph DB path: explicit override > the decomp manifest's recorded db_path.
    
    The spine lives in the decomp-core graph DB; this core writes its overlay into
    the SAME DB (a shared substrate resource owned by no single core -- pass-2
    graph-DB-ownership evidence).
    """
```

```python
def compute_worklist(
    segments: List[SpineSegment],                       # Ordered layer-0 spine
    review_markers: Dict[str, str],                     # segment_id -> decision (persisted)
    variants: Optional[Dict[str, Dict[str, str]]] = None,  # segment_id -> {transcriber: text} (intra-graph)
) -> List[WorklistItem]:  # Items still needing review, flagged
    """
    Recompute the worklist from layer-0 + signals + review state (only decisions persist).
    
    Segments already decided in this session (reviewed/corrected/skipped) drop out;
    everything flagged by a deterministic signal and not yet decided surfaces.
    """
```

```python
def confirm_seam(
    seam: str,                 # Seam label
    summary_lines: List[str],  # What the operator is accepting
    warnings: List[str],       # Tier-1 warnings
    assume_yes: bool = False,  # Headless: accept without prompting
) -> bool:  # True = proceed, False = aborted
    """
    HITL approval seam in its cheapest viable form (log + optional CLI prompt).
    
    Per-seam HITL-assist annotation (5 fields):
      1. signal: per-document summary + Tier-1 worklist flags
      2. deterministic pre-filter: compute_signal_flags (no AI)
      3. modality-bridge candidate: spectrogram / audio review (future Tier 2)
      4. authoritative verifier: re-transcribe-and-compare (future Tier 3; Gemini)
      5. flywheel capture: decisions persist as graph nodes/edges (DURABLE, unlike
         decomp's log-only seam -- the correction overlay IS the captured decision)
    input() blocks the loop; safe between stages with no jobs in flight (pass-2).
    """
```

```python
async def prune_empty_segments(
    queue: JobQueue,        # Started job queue
    cfg: CorrectionConfig,  # Run configuration
    graph_id: str,          # Graph-storage capability id
    source_id: str,         # Source being corrected
    total_count: int,       # Total segment count (for the summary)
    session_id: str,        # Owning session id
) -> Dict[str, Any]:  # {"pruned": n, "correction_id": id|None}
    """
    First operation: prune empty (silence) segments as one grouping correction (D14).
    
    Deterministic, no-human restructure proof: loads ONLY the empty segments
    (server-side filter -- scale-shaped) under the chosen rendition spine, builds
    a batch grouping Correction + DERIVED_FROM edges, commits via the queue, and
    records REVIEWED markers (decision=corrected). Layer-0 untouched; reversible
    by superseding.
    """
```

```python
def collect_capability_info(
    manager: CapabilityManager,   # Manager holding the loaded capabilities
    instance_ids: List[str],  # Instance ids to record
) -> Dict[str, Dict[str, Any]]:  # instance_id -> {name, version, db_path}
    "Record capability identity + data-DB pointers for the run manifest (provenance)."
```

```python
def _journal_run_event(
    """
    Append a host-tier run event to the journal (CR-14 follow-up).
    
    The cores are the trusted host writer class: RUN_STARTED/RUN_FINISHED
    bracket the run so the run manifest (same run_id) links to every job row
    the run produced. No-op when the manager has no journal store; append
    failures stay LOUD (journal contract).
    """
```

```python
async def run_correction(
    manager: CapabilityManager,            # Manager with the graph capability loaded
    queue: JobQueue,                   # Started job queue
    cfg: CorrectionConfig,             # Run configuration
    decomp_manifest_path: str,         # Decomp run manifest to correct
    graph_db_path: str,                # Resolved graph DB path (shared with decomp)
    run_id: Optional[str] = None,      # Override run id
    session_id: Optional[str] = None,  # Resume/reopen an existing session
    reopen: bool = False,              # Reopen a completed session
) -> CorrectionManifest:  # Manifest of the correction run
    """
    Correct every source in a decomp run manifest (prune + worklist surfacing).
    
    Per source: load spine (with variant slices) -> recompute worklist -> prune
    empty segments [prune-review seam] -> project effective spine -> record
    outcome. The cross-transcriber diff is INTRA-GRAPH (stage 5): variant texts
    come from the segments' own Transcript slice refs — no second manifest. The
    fine spine is scoped to the chosen AudioRendition (cfg.rendition_selector;
    auto-selected when a source has one decomposed rendition). Resumable: a prior
    session's review markers drop decided segments.
    """
```

```python
def _format_worklist_item(
    "Render a worklist item for the CLI review seam (text + timing + flags + hints)."
```

```python
async def review_worklist(
    queue: JobQueue,                                    # Started job queue
    cfg: CorrectionConfig,                              # Run configuration
    source_id: str,                                     # Source under review
    worklist: List[WorklistItem],                       # Flagged, undecided items
    session_id: str,                                    # Owning session id
    variant_by_segment: Optional[Dict[str, str]] = None,  # segment_id -> divergent variant text
    max_items: int = 0,                                 # Cap (0 = all)
) -> Dict[str, int]:  # {"corrected": n, "skipped": n, "reviewed": n}
    """
    Interactive review loop -> text_content corrections (cheapest HITL seam).
    
    Per item: present text + timing + flags (+ the divergent variant text +
    cross-transcript cache hit), then read a decision from stdin:
      [a]ccept (mark reviewed) / [e]dit (commit a text_content correction;
      auto-supersedes any prior correction on the segment) / [s]kip / [q]uit.
    On 'e', the next stdin line is the new text (blank -> adopt the variant) —
    adopting a variant IS the "extract the superior lightweight reading" move,
    and its provenance is the variant slice already on the segment.
    Drivable headless via a stdin pipe (E9-companion); cfg.assume_yes marks every
    item reviewed with no edits.
    """
```

```python
async def run_review(
    manager: CapabilityManager,            # Manager with the graph capability loaded
    queue: JobQueue,                   # Started job queue
    cfg: CorrectionConfig,             # Run configuration
    decomp_manifest_path: str,         # Decomp run manifest to review
    graph_db_path: str,                # Resolved graph DB path (shared with decomp)
    run_id: Optional[str] = None,      # Override run id
    session_id: Optional[str] = None,  # Resume/reopen an existing session
    reopen: bool = False,              # Reopen a completed session
    max_items: int = 0,                # Max worklist items to review per source (0 = all)
) -> CorrectionManifest:  # Manifest of the review run
    """
    Interactive review pass over a decomp manifest's flagged worklist (text corrections).
    
    Like run_correction but enters the text-correction review loop instead of the
    prune. Empty segments are excluded from the review worklist (they belong to the
    prune); the effective spine is projected from the SOURCE's corrections (across
    sessions), so a resumed/reopened session sees prior prune + text corrections.
    The variant hints come from the segments' own slice refs (intra-graph). The
    fine spine is scoped to the chosen AudioRendition (cfg.rendition_selector).
    """
```


### signals (`signals.ipynb`)
> Pure deterministic Tier-1 signal functions (no capability calls): empty-segment

#### Import

```python
from cjm_transcript_correction_core.signals import (
    detect_empty_segments,
    boundary_punct_caps_flags,
    fa_coverage_flags,
    levenshtein,
    phonetic_key,
    variant_divergence,
    cluster_variants,
    compute_signal_flags
)
```

#### Functions

```python
def detect_empty_segments(
    segments: List[SpineSegment],  # Ordered spine segments
) -> List[int]:  # Positions (in `segments`) of empty-text segments
    "Find empty-text segments (silence VAD chunks with no aligned words; decomp D14)."
```

```python
def _ends_terminal(text: str) -> bool:  # True if text ends with sentence-terminal punctuation
    """Whether a segment's text ends with terminal punctuation (trailing quotes/brackets ignored)."""
    t = (text or "").rstrip().rstrip("\"')”’")
    return t.endswith(_TERMINAL_PUNCT)


def _starts_upper(text: str) -> bool:  # True if the first alphabetic char is uppercase
    "Whether a segment's text ends with terminal punctuation (trailing quotes/brackets ignored)."
```

```python
def _starts_upper(text: str) -> bool:  # True if the first alphabetic char is uppercase
    """Whether a segment's text starts with an uppercase letter (leading quotes/brackets ignored)."""
    for ch in (text or "").lstrip("\"'(“‘")
    "Whether a segment's text starts with an uppercase letter (leading quotes/brackets ignored)."
```

```python
def boundary_punct_caps_flags(
    segments: List[SpineSegment],  # Ordered spine segments
) -> Dict[int, List[str]]:  # segment index -> boundary flags
    """
    Bidirectional boundary punctuation/capitalization heuristics (in-segment only).
    
    At each border (seg[i] -> seg[i+1]) flag the two error directions a downstream
    grouping workflow cares about, WITHOUT ever merging across audio segments:
      - "boundary-missing-terminal": seg[i] lacks terminal punctuation and seg[i+1]
        starts uppercase -> a sentence may end here but is missing a period.
      - "boundary-terminal-then-lowercase": seg[i] ends terminal but seg[i+1] starts
        lowercase -> one sentence may have been split across the border.
    Empty neighbours are skipped (handled by the prune).
    """
```

```python
def fa_coverage_flags(
    segments: List[SpineSegment],  # Ordered spine segments
) -> Dict[int, List[str]]:  # segment index -> coverage flags
    """
    Flag segments whose forced-alignment coverage looks suspect (Tier-1).
    
    Empty-text segments (no aligned words) and segments missing source-coordinate
    timing are flagged; both are alignment-failure signals shared by text and
    segmentation errors.
    """
```

```python
def levenshtein(
    a: str,  # First string
    b: str,  # Second string
) -> int:  # Edit distance
    "Levenshtein edit distance (pure, in-core; variant-clustering primitive)."
```

```python
def phonetic_key(
    word: str,  # A single word token
) -> str:  # A coarse phonetic key (Soundex-like, in-core)
    """
    Compute a coarse phonetic key for a word (groups like-sounding variants).
    
    A lightweight Soundex-style reduction (first letter + consonant codes, vowels
    dropped): enough to bucket transcription variants of one entity for
    fix-one-fix-all, without a phonetics dependency.
    """
```

```python
def _normalize_text(text: str) -> str:  # Lowercased alphabetic word tokens, space-joined
    """Normalize segment text for cross-transcriber comparison."""
    return " ".join(_WORD_RE.findall((text or "").lower()))


def variant_divergence(
    segments: List[SpineSegment],            # Layer-0 spine (authoritative text)
    variants: Dict[str, Dict[str, str]],     # segment_id -> {transcriber: chunk text} (from the graph)
) -> Dict[int, Tuple[str, str]]:  # spine index -> (authoritative_text, first divergent variant)
    "Normalize segment text for cross-transcriber comparison."
```

```python
def variant_divergence(
    segments: List[SpineSegment],            # Layer-0 spine (authoritative text)
    variants: Dict[str, Dict[str, str]],     # segment_id -> {transcriber: chunk text} (from the graph)
) -> Dict[int, Tuple[str, str]]:  # spine index -> (authoritative_text, first divergent variant)
    """
    Within-segment cross-transcriber divergence (stage 5: intra-graph).
    
    The shared-skeleton model stores every transcriber's chunk text as a slice
    on ONE segment, so divergence is a WITHIN-NODE comparison now (C14 realized)
    — no second spine, no positional join. Proper-noun / error sites concentrate
    where the normalized texts diverge (the force-multiplier signal); the
    authoritative transcriber's own variant compares equal by construction.
    """
```

```python
def cluster_variants(
    words: List[str],    # Candidate word tokens (e.g. divergent proper nouns)
    max_edits: int = 2,  # Max edit distance to join two words into one cluster
) -> List[List[str]]:  # Clusters (size > 1) of like-sounding / near-spelled variants
    """
    Cluster word variants by phonetic key + edit distance (fix-one-fix-all).
    
    Buckets transcription variants of one entity so a single decision can map them
    all to a canonical form. Pure, in-core (no phonetics dependency).
    """
```

```python
def compute_signal_flags(
    segments: List[SpineSegment],                       # Ordered layer-0 spine
    variants: Optional[Dict[str, Dict[str, str]]] = None,  # segment_id -> {transcriber: text} (intra-graph)
) -> Dict[int, List[str]]:  # segment index -> combined Tier-1 flags
    """
    Combine all deterministic Tier-1 signals into per-segment flags.
    
    The worklist is RECOMPUTED from this each session (only decisions persist);
    new signals join here and are picked up automatically. Stage 5: the
    transcriber-divergence signal reads the segments' own variant slices
    (intra-graph), not a second decomp spine.
    """
```

#### Variables

```python
_TERMINAL_PUNCT  # incl. CJK full-stop / ! / ?
_WORD_RE  # alphabetic word tokens (unicode-aware)
```